# Import libraries

In [1]:
import uuid
import copy
from datetime import datetime
from pathlib import Path
import getpass
import sys
import pandas as pd
import numpy as np
import json
import pickle
import re
import os
import glob
from typing import Dict, List, Optional, Any, Tuple
from tqdm.notebook import tqdm
import duckdb

sys.path.append('/Users/BKieft/Metabolomics/metatlas2')
import metatlas2.database_interact as dbi
import metatlas2.lcmsruns_tools as lrt
import metatlas2.ms1_ms2_analysis as msa
import metatlas2.rt_align_tools as rat

pd.options.display.max_colwidth = 300

# Define project variables

In [2]:
# Project setup
DATA_DIR = Path("/Users/BKieft/Metabolomics/metatlas2/data/")
DATABASES_DIR = DATA_DIR / "databases"
PROJECT = "20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519"
PROJECT_DIR = DATA_DIR / "test_data/projects" / PROJECT

# QC Files Configuration
QC_FILE_PATH = PROJECT_DIR / "lcmsruns"  # Base path to search for QC files
MIN_QC_FILES = 1  # Minimum number of QC files required for RT correction

# Database Configuration
DATABASE_PATH = "/Users/BKieft/Metabolomics/metatlas2/data/databases/compounds/metatlas.duckdb"
PROJECT_DB_PATH = PROJECT_DIR / f"{PROJECT}.duckdb"  # Project-specific database

# Atlas Configuration
# QC (to make model)
QC_ATLAS_NAME = "HILIC Positive QC Default"  # Atlas name for QC compounds
QC_ATLAS_UID = "atlas-65c9223bd8be48cb820a0130ed8939bd"  # Optional: specify atlas UID directly
# Target (to be aligned by model)
TARGET_ATLAS_NAME = "HILIC Positive ISTD Default"  # or
TARGET_ATLAS_UID = "atlas-fbceed0e6d8b4c349d8fcc81cf0fe9fd"

# Mass Tolerance Settings
DEFAULT_MZ_TOLERANCE_PPM = 20.0  # Default m/z tolerance in ppm for compound matching
RT_WINDOW_EXPANSION = 1  # Extra RT window (minutes) around Atlas RT for QC peak search
MIN_PEAK_INTENSITY = 1e5  # Minimum peak intensity for QC peak consideration

# Polynomial Model Settings - ALWAYS USE SECOND-ORDER POLYNOMIAL
POLYNOMIAL_DEGREE = 2  # Fixed second-order polynomial (quadratic) for RT correction
MIN_OBSERVATIONS_PER_COMPOUND = 1 # Minimum observations (files) per compound to be included in RT correction model
MIN_COMPOUNDS_FOR_MODELING = 2  # Minimum compounds needed to build RT correction model
EXCLUDE_INCHIKEYS = ["OVRNDRQMDRJTHS-ZEUBEQSHSA-N"] # Optional: compounds from QC to exclude from RT alignment model
R2_THRESHOLD = 0.7  # Minimum R-squared for acceptable model performance
MAX_RESIDUAL_RT = 1  # Maximum acceptable residual RT difference (minutes)

# Output Configuration
TIMESTAMP = datetime.now().isoformat()  # Timestamp for output files
ANALYST_NAME = getpass.getuser()

# Organize settings
METADATA = {
    "timestamp": TIMESTAMP,
    "analyst": ANALYST_NAME
}
DATA_EXTRACT_SETTINGS = {
    "default_ppm_tolerance": DEFAULT_MZ_TOLERANCE_PPM,
    "window_expansion": RT_WINDOW_EXPANSION,
    "min_peak_intensity": MIN_PEAK_INTENSITY,
}
RT_SETTINGS = {
    "polynomial_degree": POLYNOMIAL_DEGREE,
    "min_observations_per_compound": MIN_OBSERVATIONS_PER_COMPOUND,
    "min_compounds_for_modeling": MIN_COMPOUNDS_FOR_MODELING,
    "exclude_inchikeys": EXCLUDE_INCHIKEYS,
    "r2_threshold": R2_THRESHOLD,
    "max_residual_rt": MAX_RESIDUAL_RT
}

# Create Project Database

In [3]:
# Create project-specific database
dbi.create_project_database(PROJECT_DB_PATH, overwrite_existing=True)

Overwrite is set to True; Deleted existing database at /Users/BKieft/Metabolomics/metatlas2/data/test_data/projects/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519.duckdb and proceeding...
Project database created at /Users/BKieft/Metabolomics/metatlas2/data/test_data/projects/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519.duckdb


# Load or save LCMS runs

In [4]:
# Save LCMS runs to project database
lcmsrun_files = dbi.save_lcmsruns_to_db(
    project_db_path=PROJECT_DB_PATH,
    project_path=PROJECT_DIR,
    project_metadata=METADATA,
    overwrite_existing=True
)

Saved 8 LCMS runs to database:

Chromatography: HILICZ
  Polarity: FPS
    experimental: 0 files
    qc: 2 files
    injbl: 1 files
    exctrl: 0 files
    istd: 1 files
  Polarity: POS
    experimental: 3 files
    qc: 0 files
    injbl: 0 files
    exctrl: 1 files
    istd: 0 files


# Extract and Match QC Compounds

In [5]:
matches_df, matching_stats = msa.extract_and_match_qc_compounds(
    project_db_path=PROJECT_DB_PATH,
    database_path=DATABASE_PATH,
    qc_atlas_uid=QC_ATLAS_UID,
    metadata=DATA_EXTRACT_SETTINGS
)

Loading QC files and atlas compounds from databases...
Retrieved 20 compounds for atlas atlas-65c9223bd8be48cb820a0130ed8939bd
Atlas chromatography: HILICZ, polarity: positive
Found 2 QC files and 20 QC compounds
Extracting MS1 data from QC files...


Extracting MS1 data:   0%|          | 0/2 [00:00<?, ?it/s]

  Extracted 44810 peaks from 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_FPS_MS1_2_Winter-RWi_3_Rg70to1050-CE102040norm-rhizo-QC_Run58.h5 (ms1_pos)
  Extracted 19240 peaks from 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_FPS_MS1_2_Winter-RWi_3_Rg70to1050-CE102040norm-rhizo-QC_Run58.h5 (ms1_neg)
  Extracted 42533 peaks from 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_FPS_MS1_33_Summer-VSu_6_Rg70to1050-CE102040norm-veg-core-QC_Run107.h5 (ms1_pos)
  Extracted 19417 peaks from 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_FPS_MS1_33_Summer-VSu_6_Rg70to1050-CE102040norm-veg-core-QC_Run107.h5 (ms1_neg)
Total MS1 peaks extracted: 126,000

Matching QC compounds to extracted peaks...


Matching compounds:   0%|          | 0/20 [00:00<?, ?it/s]


Matching completed:
  Compounds with matches: 20/20
  Total compound-file matches: 40
  Mean m/z error: -0.21 ± 0.51 ppm
  Mean RT difference: 0.028 ± 0.119 min


# Build RT Alignment Model

In [6]:
# Build RT alignment model using QC compound matches
best_model, modeling_results_df, compound_rt_stats = rat.build_rt_alignment_model(matches_df, RT_SETTINGS)

Building RT alignment model...
Filtered out 2 matches by InChI Key.

RT Statistics Summary:
  Atlas RT range: 1.09 - 17.01 min
  Observed RT range (median): 1.25 - 17.14 min
  Mean RT difference (observed - atlas): 0.016 ± 0.109 min
Using 19 compounds with ≥1 observations (QC files) for modeling
Building polynomial model (degree 2)...

Model built successfully:
  Model type: Polynomial degree 2
  R² = 0.9998
  RMSE = 0.0820 min
  Max residual = 0.2197 min
  Equation: RT_corrected = 0.188479 + 0.946110 * RT_atlas + 0.002824 * RT_atlas^2

Compound RT Statistics:


,compound_name,inchi_key,ref_rt_peak,exp_rt_median,rt_diff_median,observation_count,exp_rt_std
0,"tyrosine (U - 13C, 15N)",OUYCCCASQSFEME-CMLFETTRSA-N,11.8552,11.751900,-0.1032,2,0.0209
1,"alanine (U - 13C, 15N)",QNAYBMKLOCPYGJ-UVYXLFMMSA-N,13.4051,13.358000,-0.0471,2,0.0095
2,"cystine (U - 13C, 15N)",LEVWYRKDKASIDU-OGYFDXEDSA-N,16.9043,17.003099,0.0988,2,0.0066
3,"threonine (U - 13C, 15N)",AYFVYJQAPQTCCC-KJQIOZMZSA-N,13.4896,13.384000,-0.1055,2,0.0094
4,"lysine (U - 13C, 15N)",KDXKERNSBIXSRK-JMKXWGMHSA-N,17.0113,17.136400,0.1251,2,0.0051
5,"valine (U - 13C, 15N)",KZSNJWFQEVHDMF-XAFSXMPTSA-N,11.1160,11.014900,-0.1011,2,0.0159
6,"tryptophan (U - 13C, 15N)",QIVBCDIJIAJPQS-HNEHNLBWSA-N,10.1566,10.180700,0.0240,2,0.0109
7,"methionine (U - 13C, 15N)",FFEARJCKVFRZRR-XAFSXMPTSA-N,10.4410,10.409400,-0.0315,2,0.0075
8,ABMBA (unlabeled),LCMZECCEEOQWLQ-UHFFFAOYSA-N,1.0938,1.253400,0.1596,2,0.0001
9,"aspartic acid (U - 13C, 15N)",CKLJMWTZIZZHCS-NYTNKSBQSA-N,16.1304,16.169500,0.0391,2,0.0116


# Create RT-Corrected Experimental Data and Save to Database

In [7]:
rt_summary = rat.apply_rt_correction_to_target(
                PROJECT_DB_PATH,
                DATABASE_PATH,
                TARGET_ATLAS_UID,
                best_model,
                lcmsrun_files,
                modeling_results_df,
                ANALYST_NAME,
                TIMESTAMP
            )

Cloning target atlas and applying RT correction...
Retrieved atlas metadata for: HILIC Positive ISTD Default
Retrieved 65 compounds for atlas atlas-fbceed0e6d8b4c349d8fcc81cf0fe9fd
Atlas chromatography: HILICZ, polarity: positive
RT alignment model saved to database with UID: rta-f8e39a9bba124e10996a79636742b7ac
Created RT-corrected atlas with 65 associations.
RT correction completed: 65/65 compounds corrected
Correction statistics: mean = 0.0025, std = 0.0657 min

RT-corrected atlas created:
  Atlas UID: atlas-rt-e644fbc10d674744afd8d4b557879af5
  Atlas name: HILIC Positive ISTD Default (RT Corrected)
  RT alignment model UID: rta-f8e39a9bba124e10996a79636742b7ac
  Project database: /Users/BKieft/Metabolomics/metatlas2/data/test_data/projects/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519.duckdb


In [8]:
atlas_info = dbi.list_available_atlases(str(PROJECT_DB_PATH))
if not atlas_info.empty:
    print(f"   Available atlases:")
    for _, row in atlas_info.iterrows():
        print(f"      {row['atlas_uid']}")
        print(f"            {row['atlas_name']}")
        print(f"            {row['chromatography']} {row['polarity']}")
        print(f"            {row['compound_count']} compounds")
        print(f"            {row['creation_time']}")
else:
    print(f"\n   No atlases found")

   Available atlases:
      atlas-rt-e644fbc10d674744afd8d4b557879af5
            HILIC Positive ISTD Default (RT Corrected)
            HILICZ positive
            65 compounds
            2025-08-26 16:30:00.596011


In [15]:
rt_correction_entry = dbi.get_rt_correction_table_entry(PROJECT_DB_PATH, 'atlas-rt-e644fbc10d674744afd8d4b557879af5')
pd.DataFrame([rt_correction_entry])

,rt_alignment_uid,project_name,atlas_uid,model_type,polynomial_degree,r_squared,rmse,coefficients,equation,qc_files_count,compounds_used_count,created_by,creation_time,model_metadata
0,rta-f8e39a9bba124e10996a79636742b7ac,20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519,atlas-rt-e644fbc10d674744afd8d4b557879af5,polynomial,2,0.99977,0.081997,"[0.0, 0.9461100910200734, 0.0028237813678847904]",RT_corrected = 0.188479 + 0.946110 * RT_atlas + 0.002824 * RT_atlas^2,2,19,BKieft,2025-08-26 16:30:00.596011,"{""qc_files"": [""20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_FPS_MS1_2_Winter-RWi_3_Rg70to1050-CE102040norm-rhizo-QC_Run58.h5"", ""20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_FPS_MS1_33_Summer-VSu_6_Rg70to1050-CE102040norm-veg-core-QC_Run107.h5""], ""compounds..."
